# Filtrado y consolidación de datos de laboratorio (Senasa)

Notebook reordenado por etapas a partir del original. Además del reordenamiento por etapas, respecto de la versión original: las rutas de los Excel se resuelven solas en cualquier sistema operativo, todos los `merge` llevan `validate="one_to_one"` (cortan con error si aparece una clave repetida, en vez de multiplicar filas en silencio), y se agregaron las etapas 1b y 5.

Etapas:
1. Carga de datos desde Excel.
1b. Conversión de `Fecha de Recepción` a fecha real.
2. Pivoteo de las tablas con múltiples análisis por identificador.
3. Fusión secuencial de todas las tablas en un único dataset.
4. Limpieza final: fusión de columnas duplicadas por el merge.
5. Separación de la `Descripción` en metadatos (lote, fechas, muestra, etc.).


## Etapa 1 — Carga de datos

Se leen los reportes de Senasa (cenizas, humedad, proteínas, grasa, sólidos, dulce de leche, lactosa y galactosa) desde los archivos Excel y se descarta la columna `Análisis`, que no aporta información.

In [1]:
# Carga de todos los reportes desde sus archivos Excel y eliminación de la
# columna 'Análisis' (no aporta información).
#
# La carpeta de datos se resuelve sola: se prueba cada ruta candidata y se usa
# la primera que exista, así el notebook corre igual en Windows, Linux o macOS
# sin editar el código. Para agregar una máquina nueva, sumar su ruta a la lista.

import pandas as pd
from pathlib import Path

CANDIDATAS = [
    Path("C:/proyecto/datos"),
    Path("datos"),
    Path("../datos"),
]

BASE = next((p for p in CANDIDATAS if p.is_dir()), None)
if BASE is None:
    raise FileNotFoundError(
        "No se encontró la carpeta de datos. Rutas probadas:\n  "
        + "\n  ".join(str(p.resolve()) for p in CANDIDATAS)
    )
print("Carpeta de datos:", BASE.resolve())

ORDEN = ["cen", "hum", "prot", "grasa", "sol", "ddl", "lac", "gal"]
ARCHIVOS = {
    "cen":   "Reporte Filtro por Muestra 2_2024-06-01_2026-08-20_Oficial Senasa_CENIZAS.xlsx",
    "hum":   "Reporte Filtro por Muestra 2_2024-01-01_2026-08-20_Oficial Senasa_humedad.xlsx",
    "prot":  "Reporte Filtro por Muestra 2_2024-06-01_2026-08-20_Oficial Senasa_PROTEINAS.xlsx",
    "grasa": "Reporte Filtro por Muestra 2_2024-06-01_2026-08-20_Oficial Senasa_GRASA.xlsx",
    "sol":   "Reporte Filtro por Muestra 2_2024-06-01_2026-08-20_Oficial Senasa_SOLIDOS.xlsx",
    "ddl":   "Reporte Filtro por Muestra 2_2020-01-01_2026-08-24_FQ_dulce de leche.xlsx",
    "lac":   "Reporte Filtro por Muestra 2_2020-01-01_FQ_LACTOSA.xlsx",
    "gal":   "Reporte Filtro por Muestra 2_2020-01-01_FQ_GALACTOSA.xlsx",
}

tablas = {}
for nombre in ORDEN:
    ruta = BASE / ARCHIVOS[nombre]
    if not ruta.exists():
        raise FileNotFoundError(f"Falta el archivo: {ruta}")
    tablas[nombre] = pd.read_excel(ruta).drop(columns=["Análisis"], errors="ignore")
    print(f"  {nombre:6} {len(tablas[nombre]):6} filas")

cen, hum, prot, grasa, sol, ddl, lac, gal = (tablas[n] for n in ORDEN)


Carpeta de datos: /home/matiassbianco/IA/datos
  cen       551 filas


  hum      2696 filas
  prot      553 filas


  grasa    4028 filas


  sol      2316 filas


  ddl     17826 filas


  lac     14369 filas


  gal      4923 filas


## Etapa 1b — Conversión de fechas

`Fecha de Recepción` llega como texto en formato `dd-mm-aaaa`. Se convierte a fecha real **antes** de las fusiones, indicando el formato de forma explícita para que el día no se lea como mes. Es también una de las cinco claves de merge, así que tiene que quedar igual en las ocho tablas.

In [2]:
# Conversión de 'Fecha de Recepción' a fecha real.
#
# En los Excel viene como TEXTO en formato día-mes-año ('19-08-2026'). Si se
# convierte sin indicar el formato, pandas puede leer el día como mes y dar
# vuelta las fechas en silencio, así que se pasa el formato explícito.
#
# Se convierte ANTES de las fusiones, para que la clave de merge sea del mismo
# tipo en todas las tablas.

FORMATO_FECHA = "%d-%m-%Y"


def convertir_fecha(df, nombre, columna="Fecha de Recepción"):
    """Convierte la columna de fecha a datetime avisando qué no se pudo convertir."""
    original = df[columna]
    convertida = pd.to_datetime(original, format=FORMATO_FECHA, errors="coerce")

    vacias = original.isna().sum()
    perdidas = convertida.isna() & original.notna()

    aviso = ""
    if vacias:
        aviso += f"  {vacias} vacías en el origen"
    if perdidas.any():
        aviso += f"  [ojo] {perdidas.sum()} no coinciden con {FORMATO_FECHA}: "
        aviso += ", ".join(map(str, original[perdidas].unique()[:5]))
    print(f"  {nombre:6} {convertida.notna().sum():6} fechas válidas{aviso}")

    df = df.copy()
    df[columna] = convertida
    return df


for nombre in ORDEN:
    tablas[nombre] = convertir_fecha(tablas[nombre], nombre)

cen, hum, prot, grasa, sol, ddl, lac, gal = (tablas[n] for n in ORDEN)

# Las filas con fecha vacía quedan como NaT y, igual que antes cuando eran
# texto vacío, nunca van a coincidir con nada en los merges: NaT != NaT.
print("\nRango de fechas de recepción:")
todas = pd.concat([tablas[n]["Fecha de Recepción"] for n in ORDEN])
print(f"  desde {todas.min():%d/%m/%Y} hasta {todas.max():%d/%m/%Y}")

  cen       551 fechas válidas
  hum      2696 fechas válidas
  prot      553 fechas válidas
  grasa    4028 fechas válidas
  sol      2316 fechas válidas
  ddl     17826 fechas válidas
  lac     14369 fechas válidas
  gal      4923 fechas válidas

Rango de fechas de recepción:
  desde 04/01/2020 hasta 28/08/2026


## Etapa 2 — Pivoteo de tablas con múltiples análisis por identificador

Grasa, sólidos, dulce de leche, lactosa y galactosa vienen en formato "largo" (una fila por análisis). Las siguientes celdas aplican el mismo procedimiento a cada una: quitan columnas `Unnamed`, separan las columnas fijas de las que se pivotean, y arman una tabla con una sola fila por `Identificador`, con columnas `Resultado ...` y `Técnica ...` por cada análisis.

In [3]:
# Pivoteo de GRASA: una fila por Identificador, con una columna
# Resultado/Técnica por cada análisis de materia grasa.

grasa = grasa.loc[:, ~grasa.columns.str.startswith("Unnamed")]
# ============================================================
# 3. VER LAS COLUMNAS
# ============================================================
print("Columnas:")
print(grasa.columns.tolist())
# ============================================================
# 4. SEPARAR COLUMNAS FIJAS (no cambian por Identificador)
#    de las columnas que se van a "pivotear"
# ============================================================
columnas_pivot = ["Nombre del Análisis", "Técnica", "Resultado"]
columnas_fijas = [c for c in grasa.columns if c not in columnas_pivot and c != "Identificador"]

fijas = grasa[["Identificador"] + columnas_fijas].drop_duplicates(subset="Identificador")
# ============================================================
# 5. CREAR LA TABLA CON UN IDENTIFICADOR POR FILA
#    (pivotea Resultado Y Técnica, agregando columnas nuevas por análisis)
# ============================================================
pivote = grasa.pivot_table(
    index="Identificador",
    columns="Nombre del Análisis",
    values=["Resultado", "Técnica"],
    aggfunc="first",
)

# Aplanar encabezados de dos niveles -> "Resultado Materia Grasa g/ 100g (*)"
pivote.columns = [f"{valor} {analisis}" for valor, analisis in pivote.columns]
pivote = pivote.reset_index()
# ============================================================
# 6. UNIR: columnas fijas + columnas pivoteadas, por Identificador
# ============================================================
tabla = fijas.merge(pivote, on="Identificador", how="left", validate="one_to_one")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
print("\nTABLA NUEVA:")
print(tabla.columns.tolist())
print(tabla.head())

gras = tabla

Columnas:
['Cliente', 'Envío', 'Identificador', 'Descripción', 'Nombre del Análisis', 'Técnica', 'Resultado', 'Fecha de Recepción']

TABLA NUEVA:
['Identificador', 'Cliente', 'Envío', 'Descripción', 'Fecha de Recepción', 'Resultado Materia Grasa g/ 100g (*)', 'Resultado Materia grasa en extracto seco g/ 100g (*)', 'Técnica Materia Grasa g/ 100g (*)', 'Técnica Materia grasa en extracto seco g/ 100g (*)']
  Identificador                            Cliente    Envío  \
0   MO-11492-01                       IND-21        O-11492   
1   MO-11482-05  IND-51                             O-11482   
2   MO-11482-04  IND-51                             O-11482   
3   MO-11482-03  IND-51                             O-11482   
4   MO-11482-02  IND-51                             O-11482   

                                         Descripción Fecha de Recepción  \
0  FIAMBRE COCIDO PARA PALETA DE CERDO  - LOTE 07...         2026-08-20   
1        QUESO AZUL EN PORCIONES - Lote  212-4  - M5         202

In [4]:
# Pivoteo de SÓLIDOS: mismo procedimiento que en la celda anterior,
# aplicado a la tabla de sólidos.

sol = sol.loc[:, ~sol.columns.str.startswith("Unnamed")]
# ============================================================
# 3. VER LAS COLUMNAS
# ============================================================
print("Columnas:")
print(sol.columns.tolist())
# ============================================================
# 4. SEPARAR COLUMNAS FIJAS (no cambian por Identificador)
#    de las columnas que se van a "pivotear"
# ============================================================
columnas_pivot = ["Nombre del Análisis", "Técnica", "Resultado"]
columnas_fijas = [c for c in sol.columns if c not in columnas_pivot and c != "Identificador"]

fijas = sol[["Identificador"] + columnas_fijas].drop_duplicates(subset="Identificador")
# ============================================================
# 5. CREAR LA TABLA CON UN IDENTIFICADOR POR FILA
#    (pivotea Resultado Y Técnica, agregando columnas nuevas por análisis)
# ============================================================
pivote = sol.pivot_table(
    index="Identificador",
    columns="Nombre del Análisis",
    values=["Resultado", "Técnica"],
    aggfunc="first",
)

# Aplanar encabezados de dos niveles -> "Resultado Materia Grasa g/ 100g (*)"
pivote.columns = [f"{valor} {analisis}" for valor, analisis in pivote.columns]
pivote = pivote.reset_index()
# ============================================================
# 6. UNIR: columnas fijas + columnas pivoteadas, por Identificador
# ============================================================
soli = fijas.merge(pivote, on="Identificador", how="left", validate="one_to_one")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
print("\nTABLA NUEVA:")
print(soli.columns.tolist())
print(soli.head())

Columnas:
['Cliente', 'Envío', 'Identificador', 'Descripción', 'Nombre del Análisis', 'Técnica', 'Resultado', 'Fecha de Recepción']

TABLA NUEVA:
['Identificador', 'Cliente', 'Envío', 'Descripción', 'Fecha de Recepción', 'Resultado Acidez (ml NaOH 0,1 N/ 10 g sólidos no grasos) (*)', 'Resultado Sólidos Totales g / 100g (*)', 'Resultado Sólidos Totales g/ 100g (*)', 'Resultado Sólidos totales g/ 100g', 'Técnica Acidez (ml NaOH 0,1 N/ 10 g sólidos no grasos) (*)', 'Técnica Sólidos Totales g / 100g (*)', 'Técnica Sólidos Totales g/ 100g (*)', 'Técnica Sólidos totales g/ 100g']
  Identificador                            Cliente    Envío  \
0   MO-11482-05  IND-51                             O-11482   
1   MO-11482-04  IND-51                             O-11482   
2   MO-11482-03  IND-51                             O-11482   
3   MO-11482-02  IND-51                             O-11482   
4   MO-11482-01  IND-51                             O-11482   

                                   Descr

In [5]:
# Pivoteo de DULCE DE LECHE: mismo procedimiento, agregando además el
# reordenamiento de columnas para que cada Resultado quede junto a su
# Técnica correspondiente. Resultado: 'dudele'.

ddle = ddl.loc[:, ~ddl.columns.str.startswith("Unnamed")]
# ============================================================
# 3. VER LAS COLUMNAS
# ============================================================
print("Columnas:")
print(ddle.columns.tolist())
# ============================================================
# 4. SEPARAR COLUMNAS FIJAS (no cambian por Identificador)
#    de las columnas que se van a "pivotear"
# ============================================================
columnas_pivot = ["Nombre del Análisis", "Técnica", "Resultado"]
columnas_fijas = [c for c in ddle.columns if c not in columnas_pivot and c != "Identificador"]

fijas = ddle[["Identificador"] + columnas_fijas].drop_duplicates(subset="Identificador")
# ============================================================
# 5. CREAR LA TABLA CON UN IDENTIFICADOR POR FILA
#    (pivotea Resultado Y Técnica, agregando columnas nuevas por análisis)
# ============================================================
pivote = ddle.pivot_table(
    index="Identificador",
    columns="Nombre del Análisis",
    values=["Resultado", "Técnica"],
    aggfunc="first",
)

# NUEVO: reordena para que, por cada análisis, Resultado y Técnica
# queden uno al lado del otro (en vez de agrupar todos los Resultado
# primero y todas las Técnica después)
pivote = pivote.sort_index(axis=1, level=1, sort_remaining=False)

# Aplanar encabezados de dos niveles -> "Resultado Materia Grasa g/ 100g (*)"
pivote.columns = [f"{valor} {analisis}" for valor, analisis in pivote.columns]
pivote = pivote.reset_index()
# ============================================================
# 6. UNIR: columnas fijas + columnas pivoteadas, por Identificador
# ============================================================
tabla = fijas.merge(pivote, on="Identificador", how="left", validate="one_to_one")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)


dudele = tabla
dudele

Columnas:
['Cliente', 'Envío', 'Identificador', 'Descripción', 'Nombre del Análisis', 'Técnica', 'Resultado', 'Fecha de Recepción']


,Identificador,Cliente,Envío,Descripción,Fecha de Recepción,Resultado Acidez (% Ácido Láctico) (*),Técnica Acidez (% Ácido Láctico) (*),Resultado Acidez (g ácido láctico/ 100g) (*),Técnica Acidez (g ácido láctico/ 100g) (*),Técnica Acidez (g ácido oleico/100gr de muestra) (*),Resultado Acido Sórbico mg/ Kg (*),Técnica Acido Sórbico mg/ Kg (*),Resultado Actividad de agua (% Aw) (*),Técnica Actividad de agua (% Aw) (*),Resultado Aflatoxina Total ppb (*),Técnica Aflatoxina Total ppb (*),Resultado Alcalinidad de la Ceniza ml/ 100gr (*),Técnica Alcalinidad de la Ceniza ml/ 100gr (*),Resultado Almidón Total g/ 100g (*),Técnica Almidón Total g/ 100g (*),Resultado Arsénico ppm (*),Técnica Arsénico ppm (*),Resultado Azucares totales g/ 100gr (*),Técnica Azucares totales g/ 100gr (*),Resultado Azucares totales g/100gr (*),Técnica Azucares totales g/100gr (*),Resultado Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa),Técnica Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa),Resultado Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa/galactosa),Técnica Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa/galactosa),Resultado Azúcares Totales % (*),Técnica Azúcares Totales % (*),Resultado Cadmio ppm (*),Técnica Cadmio ppm (*),Resultado Calcio mg/ 100g (*),Técnica Calcio mg/ 100g (*),Resultado Calcio ppm (*),Técnica Calcio ppm (*),Resultado Carbohidratos Totales g/ 100g (*),Técnica Carbohidratos Totales g/ 100g (*),Resultado Cenizas g/ 100g (*),Técnica Cenizas g/ 100g (*),Resultado Cobre ppm (*),Técnica Cobre ppm (*),Resultado Colesterol mg% (*),Técnica Colesterol mg% (*),Resultado Colesterol mg/ 100g (*),Técnica Colesterol mg/ 100g (*),Resultado D - Galactosa g/ 100g (*),Técnica D - Galactosa g/ 100g (*),Resultado Enterotoxinas Estafilocócicas (*),Técnica Enterotoxinas Estafilocócicas (*),Resultado Fibra Alimentaria Total g/ 100g (*),Técnica Fibra Alimentaria Total g/ 100g (*),Resultado Fósforo mg/ 100 g (*),Técnica Fósforo mg/ 100 g (*),Resultado Galactosa g/ 100g (*),Técnica Galactosa g/ 100g (*),Resultado Gluten (*),Técnica Gluten (*),Resultado Gluten mg/ Kg (*),Técnica Gluten mg/ Kg (*),Resultado Grado Brix (°Bx) (*),Técnica Grado Brix (°Bx) (*),Resultado Grasas monoinsaturadas g% (*),Técnica Grasas monoinsaturadas g% (*),Resultado Grasas poliinsaturadas g% (*),Técnica Grasas poliinsaturadas g% (*),Resultado Grasas saturadas g% (*),Técnica Grasas saturadas g% (*),Resultado Grasas trans g% (*),Técnica Grasas trans g% (*),Resultado Hidratos de Carbono % (*),Técnica Hidratos de Carbono % (*),Resultado Hierro ppm (*),Técnica Hierro ppm (*),Resultado Huevo entero en polvo mg/ Kg (*),Técnica Huevo entero en polvo mg/ Kg (*),Resultado Humedad g/ 100g,Técnica Humedad g/ 100g,Resultado Humedad g/ 100g (*),Técnica Humedad g/ 100g (*),Resultado Lactosa g/ 100g (*),Técnica Lactosa g/ 100g (*),Resultado Materia Grasa g/ 100g (*),Técnica Materia Grasa g/ 100g (*),Resultado Materia grasa en extracto seco g/ 100g (*),Técnica Materia grasa en extracto seco g/ 100g (*),Resultado Melamina mg/ kg (*),Técnica Melamina mg/ kg (*),Resultado Mercurio ppm (*),Técnica Mercurio ppm (*),Resultado Pesticidas organoclorados mg/ L (*),Técnica Pesticidas organoclorados mg/ L (*),Resultado Pesticidas organofosforados mg/ L (*),Técnica Pesticidas organofosforados mg/ L (*),Resultado Piretroides mg/ L (*),Técnica Piretroides mg/ L (*),Resultado Plomo ppm (*),Técnica Plomo ppm (*),Resultado Potasio mg/100g (*),Técnica Potasio mg/100g (*),Resultado Proteína de soja mg/ Kg (*),Técnica Proteína de soja mg/ Kg (*),Resultado Proteínas g/ 100g,Técnica Proteínas g/ 100g,Resultado Proteínas g/ 100g (*),Técnica Proteínas g/ 100g (*),Resultado Proteínas g/ 100g (*),Técnica Proteínas g/ 100g (*),Resultado Selenio ppm (*),Técnica Selenio ppm (*),Resultado Sodio mg /100g (*),Técnica Sodio mg /100g (*),Resultado Solubilidad % (*),Técnica Solubilidad % (*),Resultado Sulfitos mg SO₂/ 100 gr (*),Técni

In [6]:
# Pivoteo de LACTOSA: mismo procedimiento que en dulce de leche.
# Resultado: 'lactosa'.

lact = lac.loc[:, ~lac.columns.str.startswith("Unnamed")]
# ============================================================
# 3. VER LAS COLUMNAS
# ============================================================
print("Columnas:")
print(lact.columns.tolist())
# ============================================================
# 4. SEPARAR COLUMNAS FIJAS (no cambian por Identificador)
#    de las columnas que se van a "pivotear"
# ============================================================
columnas_pivot = ["Nombre del Análisis", "Técnica", "Resultado"]
columnas_fijas = [c for c in lact.columns if c not in columnas_pivot and c != "Identificador"]

fijas = lact[["Identificador"] + columnas_fijas].drop_duplicates(subset="Identificador")
# ============================================================
# 5. CREAR LA TABLA CON UN IDENTIFICADOR POR FILA
#    (pivotea Resultado Y Técnica, agregando columnas nuevas por análisis)
# ============================================================
pivote = lact.pivot_table(
    index="Identificador",
    columns="Nombre del Análisis",
    values=["Resultado", "Técnica"],
    aggfunc="first",
)

# NUEVO: reordena para que, por cada análisis, Resultado y Técnica
# queden uno al lado del otro (en vez de agrupar todos los Resultado
# primero y todas las Técnica después)
pivote = pivote.sort_index(axis=1, level=1, sort_remaining=False)

# Aplanar encabezados de dos niveles -> "Resultado Materia Grasa g/ 100g (*)"
pivote.columns = [f"{valor} {analisis}" for valor, analisis in pivote.columns]
pivote = pivote.reset_index()
# ============================================================
# 6. UNIR: columnas fijas + columnas pivoteadas, por Identificador
# ============================================================
tablan = fijas.merge(pivote, on="Identificador", how="left", validate="one_to_one")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)


lactosa = tablan
lactosa

Columnas:
['Cliente', 'Envío', 'Identificador', 'Descripción', 'Nombre del Análisis', 'Técnica', 'Resultado', 'Fecha de Recepción']


,Identificador,Cliente,Envío,Descripción,Fecha de Recepción,Resultado Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa),Técnica Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa),Resultado Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa/galactosa),Técnica Azucares totales g/100gr (*) (fructosa/glucosa/sacarosa/maltosa/lactosa/galactosa),Resultado Contenido de Lactosa % (g/ 100ml) (*),Técnica Contenido de Lactosa % (g/ 100ml) (*),Resultado D - Galactosa g/ 100g (*),Técnica D - Galactosa g/ 100g (*),Resultado Determinación del contenido de lactosa (g/ 100ml) (*),Técnica Determinación del contenido de lactosa (g/ 100ml) (*),Resultado Galactosa g/ 100g (*),Técnica Galactosa g/ 100g (*),Resultado Lactosa g /100g (*),Técnica Lactosa g /100g (*),Resultado Lactosa g/ 100g (*),Técnica Lactosa g/ 100g (*)
0,MQ-45310-01,IND-115,Q-45310,YOGUR BEBIBLE DESCREMADO-LOTE: SVALCO- VTO: 29/09,2026-08-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,"AOAC (2005), 18th Ed. 982.14"
1,MQ-45315-01,IND-103,Q-45315,LECHE EN POLVO ENTERA- LOTE: L223S26 V8-8-2027,2026-08-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Cálculo por diferencia
2,MQ-45291-04,IND-59,Q-45291,SUERO LIQUIDO DESCREMADO- CALIBRACION N° 4,2026-08-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Cálculo por diferencia
3,MQ-45291-03,IND-59,Q-45291,SUERO LIQUIDO DESCREMADO- CALIBRACION N° 3,2026-08-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Cálculo por diferencia
4,MQ-45291-02,IND-59,Q-45291,SUERO LIQUIDO DESCREMADO- CALIBRACION N° 2,2026-08-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Cálculo por diferencia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9628,MQ-14084-05,IND-80,Q-14084,HELADO DE BANANA SPLIT,2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,"0,04",Enzymatic BioAnalysis / R-BIOPHARM,NaN,NaN,NaN,NaN,NaN,NaN,"3,70",Enzymatic BioAnalysis / R-BIOPHARM
9629,MQ-14084-04,IND-80,Q-14084,HELADO DE CHOCOLATE DOBLE TENTACIÓN,2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,"&lt; 0,03",Enzymatic BioAnalysis / R-BIOPHARM,NaN,NaN,NaN,NaN,NaN,NaN,"3,83",Enzymatic BioAnalysis / R-BIOPHARM
9630,MQ-14084-03,IND-80,Q-14084,HELADO DE DULCE DE LECHE TENTACIÓN,2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,"0,58",Enzymatic BioAnalysis / R-BIOPHARM,NaN,NaN,NaN,NaN,NaN,NaN,"4,34",Enzymatic BioAnalysis / R-BIOPHARM
9631,MQ-14084-02,IND-80,Q-14084,HELADO DE DULCE DE LECHE CLÁSICO,2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,"0,55",Enzymatic BioAnalysis / R-BIOPHARM,NaN,NaN,NaN,NaN,NaN,NaN,"4,49",Enzymatic BioAnalysis / R-BIOPHARM


In [7]:
# Pivoteo de GALACTOSA: mismo procedimiento que en las celdas anteriores.
# Resultado: 'galactosa'.

galact = gal.loc[:, ~gal.columns.str.startswith("Unnamed")]
# ============================================================
# 3. VER LAS COLUMNAS
# ============================================================
print("Columnas:")
print(galact.columns.tolist())
# ============================================================
# 4. SEPARAR COLUMNAS FIJAS (no cambian por Identificador)
#    de las columnas que se van a "pivotear"
# ============================================================
columnas_pivot = ["Nombre del Análisis", "Técnica", "Resultado"]
columnas_fijas = [c for c in galact.columns if c not in columnas_pivot and c != "Identificador"]

fijas = galact[["Identificador"] + columnas_fijas].drop_duplicates(subset="Identificador")
# ============================================================
# 5. CREAR LA TABLA CON UN IDENTIFICADOR POR FILA
#    (pivotea Resultado Y Técnica, agregando columnas nuevas por análisis)
# ============================================================
pivote = galact.pivot_table(
    index="Identificador",
    columns="Nombre del Análisis",
    values=["Resultado", "Técnica"],
    aggfunc="first",
)

# NUEVO: reordena para que, por cada análisis, Resultado y Técnica
# queden uno al lado del otro (en vez de agrupar todos los Resultado
# primero y todas las Técnica después)
pivote = pivote.sort_index(axis=1, level=1, sort_remaining=False)

# Aplanar encabezados de dos niveles -> "Resultado Materia Grasa g/ 100g (*)"
pivote.columns = [f"{valor} {analisis}" for valor, analisis in pivote.columns]
pivote = pivote.reset_index()
# ============================================================
# 6. UNIR: columnas fijas + columnas pivoteadas, por Identificador
# ============================================================
tablan = fijas.merge(pivote, on="Identificador", how="left", validate="one_to_one")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)


galactosa = tablan

Columnas:
['Cliente', 'Envío', 'Identificador', 'Descripción', 'Nombre del Análisis', 'Técnica', 'Resultado', 'Fecha de Recepción']


## Etapa 3 — Fusión secuencial de todas las tablas

Se van uniendo (`merge`) las tablas de a una, siempre por `Identificador`, `Fecha de Recepción`, `Envío`, `Descripción` y `Cliente`, hasta armar el dataset consolidado `datasos`. El orden de estas celdas se mantiene tal cual el original, porque cada fusión depende del resultado de la anterior.

In [8]:
# Fusión 1: Humedad + Cenizas -> 'ch', renombrando las columnas
# Técnica/Resultado de cada análisis para que queden identificadas.

ch = pd.merge(
    hum,
    cen,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)
ch = ch.loc[:, ~ch.columns.str.startswith("Unnamed")]
ch.rename(columns={"Técnica_y": "Técnica Cenizas g/ 100g"}, inplace=True)
ch.rename(columns={"Resultado_y": "Resultado Cenizas g/ 100g"}, inplace=True)
ch.rename(columns={"Técnica_x": "Técnica Humedad g/ 100g"}, inplace=True)
ch.rename(columns={"Resultado_x": "Resultado Humedad g/ 100g"}, inplace=True)
ch = ch.loc[:, ~ch.columns.str.startswith("Nombre del Análisis")]

In [9]:
# Fusión 2: 'ch' + Proteínas -> 'chp', renombrando columnas de Proteínas.

chp = pd.merge(
    ch,
    prot,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)
chp = chp.loc[:, ~chp.columns.str.startswith("Unnamed")]
chp.rename(columns={"Técnica": "Técnica Proteínas g/ 100g"}, inplace=True)
chp.rename(columns={"Resultado": "Resultado Proteínas g/ 100g"}, inplace=True)
chp = chp.loc[:, ~chp.columns.str.startswith("Nombre del Análisis")]

In [10]:
# Fusión 3: 'chp' + Grasa pivoteada ('gras') -> 'chpg', reordenando las
# columnas de Técnica de Materia Grasa.

chpg = pd.merge(
    chp,
    gras,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)
# chpg = chpg.loc[:, ~ch.columns.str.startswith("Unnamed")]
# chpg.rename(columns={'Técnica': 'Técnica Grasa g/ 100g'}, inplace=True)
# chpg.rename(columns={'Resultado': 'Resultado Grasa g/ 100g'}, inplace=True)
# chpg = ch.loc[:, ~ch.columns.str.startswith("Nombre del Análisis")]
tecg = chpg.pop("Técnica Materia Grasa g/ 100g (*)")
tecs = chpg.pop("Técnica Materia grasa en extracto seco g/ 100g (*)")
chpg.insert(8, "Técnica Materia Grasa g/ 100g (*)", tecg)
chpg.insert(10, "Técnica Materia grasa en extracto seco g/ 100g (*)", tecs)

In [11]:
# Fusión 4: 'chpg' + Sólidos pivoteados ('soli') -> 'chpgs', reordenando
# las columnas de Técnica de Acidez y Sólidos totales.

chpgs = pd.merge(
    chpg,
    soli,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)
tecg = chpgs.pop("Técnica Acidez (ml NaOH 0,1 N/ 10 g sólidos no grasos) (*)")
tecs = chpgs.pop("Técnica Sólidos totales g/ 100g")
chpgs.insert(12, "Técnica Acidez (ml NaOH 0,1 N/ 10 g sólidos no grasos)", tecg)
chpgs.insert(14, "Técnica Sólidos totales g/ 100g", tecs)

In [12]:
# Fusión 5: 'chpgs' + Dulce de leche pivoteado ('dudele') -> 'chpgsd'.

chpgsd = pd.merge(
    chpgs,
    dudele,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)

In [13]:
# Fusión 6: 'chpgsd' + Lactosa pivoteada ('lactosa') -> 'chpgsdl'.

chpgsdl = pd.merge(
    chpgsd,
    lactosa,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)

In [14]:
# Fusión 7 (final): 'chpgsdl' + Galactosa pivoteada ('galactosa') ->
# 'datasos', y se reubica la columna 'Fecha de Recepción' en la
# posición 4 del dataset consolidado.

datasos = pd.merge(
    chpgsdl,
    galactosa,
    on=["Identificador", "Fecha de Recepción", "Envío", "Descripción", "Cliente"],
    how="outer",
    validate="one_to_one",
)
colum = datasos.pop("Fecha de Recepción")
datasos.insert(4, "Fecha de Recepción", colum)

## Etapa 4 — Limpieza final: fusión de columnas duplicadas

Como resultado de los sucesivos `merge`, el dataset consolidado puede tener columnas de `Resultado`/`Técnica` repetidas o con sufijos `_x`/`_y`. Esta etapa define funciones auxiliares para detectar esas columnas equivalentes, fusionarlas en una sola por análisis y generar un reporte de los conflictos encontrados.

In [15]:
# Funciones auxiliares para detectar columnas 'Resultado'/'Técnica'
# equivalentes (aunque tengan sufijos _x/_y o distinto formato de
# nombre) y fusionarlas en una sola columna por análisis, informando
# los conflictos encontrados. Se aplica sobre el dataset consolidado
# 'datasos' de la Etapa 3.

import re
import unicodedata
from collections import defaultdict


def _quitar_acentos(texto: str) -> str:
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def clave_normalizada(nombre_columna: str):
    """
    Devuelve (tipo, clave) donde:
      - tipo es "Resultado" o "Técnica"
      - clave es el nombre del análisis normalizado (para poder agrupar)
    Si la columna no es de Resultado/Técnica, devuelve (None, None).
    """
    m = re.match(r"^(Resultado|Técnica)\b(.*)$", nombre_columna.strip())
    if not m:
        return None, None

    tipo, resto = m.group(1), m.group(2)

    # sacar sufijos de merge: _x, _y, _x.1, _y.2, etc. (al final del string)
    resto = re.sub(r"_(x|y)(\.\d+)?\s*$", "", resto)

    # sacar el marcador "(*)"
    resto = resto.replace("(*)", "")

    # unificar espacios alrededor de "/"
    resto = re.sub(r"\s*/\s*", "/", resto)

    # colapsar espacios múltiples y limpiar bordes
    resto = re.sub(r"\s+", " ", resto).strip(" -")

    # clave canónica: minúsculas y sin acentos, solo para comparar/agrupar
    clave = _quitar_acentos(resto.lower()).strip()

    return tipo, clave


def elegir_nombre_final(columnas_del_grupo):
    """
    Entre varias columnas que representan el mismo análisis, elige el nombre
    final: prioriza nombres sin sufijo _x/_y y, en caso de empate, el más corto.
    """

    def score(col):
        tiene_sufijo = bool(re.search(r"_(x|y)(\.\d+)?\s*$", col))
        return (tiene_sufijo, len(col))

    return sorted(columnas_del_grupo, key=score)[0]


def fusionar_columnas_duplicadas(df: pd.DataFrame, verbose: bool = True):
    """
    Devuelve (df_fusionado, reporte).
    reporte es una lista de dicts con el detalle de cada fusión realizada,
    incluyendo cuántas filas tuvieron valores en conflicto.
    """
    grupos = defaultdict(list)
    orden_grupos = []  # preserva el orden de primera aparición de cada grupo/columna

    for col in df.columns:
        tipo, clave = clave_normalizada(col)
        llave = ("__id__", col) if tipo is None else (tipo, clave)
        if llave not in grupos:
            orden_grupos.append(llave)
        grupos[llave].append(col)

    columnas_finales = {}
    reporte = []

    for llave in orden_grupos:
        tipo, clave = llave
        cols = grupos[llave]

        if tipo == "__id__":
            # columna que no es de Resultado/Técnica (Cliente, Envío, etc.)
            nombre_final = cols[0]
            columnas_finales[nombre_final] = df[nombre_final]
            continue

        if len(cols) == 1:
            columnas_finales[cols[0]] = df[cols[0]]
            continue

        nombre_final = elegir_nombre_final(cols)
        sub = df[cols]

        valores_unicos_por_fila = sub.apply(lambda fila: fila.dropna().nunique(), axis=1)
        conflictos = int((valores_unicos_por_fila > 1).sum())

        # toma, fila por fila, el primer valor no nulo según el orden de 'cols'
        fusionada = sub.bfill(axis=1).iloc[:, 0]

        columnas_finales[nombre_final] = fusionada

        reporte.append(
            {
                "tipo": tipo,
                "columna_final": nombre_final,
                "columnas_origen": cols,
                "filas_con_conflicto": conflictos,
            }
        )

    orden_final = list(columnas_finales.keys())
    resultado = pd.concat(columnas_finales.values(), axis=1, keys=orden_final)
    resultado.columns = orden_final

    if verbose:
        print(f"Columnas originales: {len(df.columns)}")
        print(f"Columnas finales:    {len(resultado.columns)}")
        print(f"Grupos fusionados:   {len(reporte)}\n")
        for item in sorted(reporte, key=lambda r: (-len(r["columnas_origen"]), r["columna_final"])):
            alerta = (
                f"  <-- {item['filas_con_conflicto']} filas en conflicto"
                if item["filas_con_conflicto"]
                else ""
            )
            print(f"[{item['tipo']}] {item['columna_final']}{alerta}")
            for c in item["columnas_origen"]:
                marca = " (elegida)" if c == item["columna_final"] else ""
                print(f"    - {c}{marca}")

    return resultado, reporte


# ------------------------------------------------------------------
# Uso: reemplazá df por tu DataFrame ya cargado
# ------------------------------------------------------------------
# df_fusionado, reporte = fusionar_columnas_duplicadas(df)


SenasaFil, reporte = fusionar_columnas_duplicadas(datasos)


/tmp/ipykernel_9332/3183655300.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fusionada = sub.bfill(axis=1).iloc[:, 0]


/tmp/ipykernel_9332/3183655300.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fusionada = sub.bfill(axis=1).iloc[:, 0]


/tmp/ipykernel_9332/3183655300.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fusionada = sub.bfill(axis=1).iloc[:, 0]


/tmp/ipykernel_9332/3183655300.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fusionada = sub.bfill(axis=1).iloc[:, 0]


/tmp/ipykernel_9332/3183655300.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fusionada = sub.bfill(axis=1).iloc[:, 0]


/tmp/ipykernel_9332/3183655300.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fusionada = sub.bfill(axis=1).iloc[:, 0]


Columnas originales: 182
Columnas finales:    136
Grupos fusionados:   24

[Resultado] Resultado Sólidos totales g/ 100g_x
    - Resultado Sólidos Totales g / 100g (*)_x
    - Resultado Sólidos Totales g/ 100g (*)_x
    - Resultado Sólidos totales g/ 100g_x (elegida)
    - Resultado Sólidos Totales g / 100g (*)_y
    - Resultado Sólidos Totales g/ 100g (*)_y
    - Resultado Sólidos totales g/ 100g_y
[Técnica] Técnica Sólidos totales g/ 100g_x
    - Técnica Sólidos totales g/ 100g_x (elegida)
    - Técnica Sólidos Totales g / 100g (*)_x
    - Técnica Sólidos Totales g/ 100g (*)_x
    - Técnica Sólidos Totales g / 100g (*)_y
    - Técnica Sólidos Totales g/ 100g (*)_y
    - Técnica Sólidos totales g/ 100g_y
[Resultado] Resultado Proteínas g/ 100g (*)
    - Resultado Proteínas g/ 100g_x
    - Resultado Proteínas g/ 100g_y
    - Resultado Proteínas g/ 100g  (*)
    - Resultado Proteínas g/ 100g (*) (elegida)
[Técnica] Técnica Proteínas g/ 100g (*)
    - Técnica Proteínas g/ 100g_x
    - Té

## Etapa 5 — Separación de la Descripción en metadatos

La `Descripción` es texto libre, pero con estructura: tramos separados por guiones, algunos etiquetados (`Lote:`, `Vto.`, `Equipo:`, `OP:`) y otros no (la marca, el nombre del cliente, un código suelto).

Se extrae **solo lo etiquetado o lo que tiene una forma inequívoca**. Lo que no se reconoce queda entero en la columna `sin_identificar`, de modo que sea visible qué quedó afuera en vez de perderse. Las marcas y nombres de cliente caen ahí a propósito: no hay forma de distinguirlos del resto del texto sin una lista previa, y suponer cuál es cuál sería inventar datos.

In [16]:
# Separación de la columna 'Descripción' en metadatos.
#
# La Descripción es texto libre, pero sigue un patrón bastante estable: tramos
# separados por guiones, algunos con etiqueta ("Lote:", "Vto.", "Equipo:") y
# otros sin ella (la marca, el nombre del cliente, un código suelto).
#
# Criterio: se extrae SOLO lo que viene etiquetado o tiene una forma inequívoca.
# Todo lo demás se guarda tal cual en 'sin_identificar', para que nada se pierda
# sin que se note. Es preferible una columna con menos datos a una columna con
# datos inventados.

import re

# Un separador es un guion con espacio de al menos un lado: así "Lote 340-32"
# no se parte al medio, pero "QUESO - Lote 340" sí.
SEPARADOR = re.compile(r"\s+[-–]\s*|\s*[-–]\s+")

FECHA = r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4}|\d{1,2}[/\-.]\d{4}|\d{1,2}[/\-.]\d{1,2}"
HORA = r"\d{1,2}[:.]\d{2}(?:[:.]\d{2})?"

# El orden importa: lo que se reconoce primero queda reservado y no lo puede
# volver a tomar otro patrón.
ETIQUETAS = [
    ("fecha_elaboracion", r"(?:elab(?:oraci[oó]n)?)\s*[:.]?\s*(" + FECHA + r")"),
    ("fecha_vencimiento", r"(?:vto|venc(?:imiento)?|vence)\s*[:.]?\s*(" + FECHA + r")"),
    ("fecha",             r"\bfecha\s*[:.]?\s*(" + FECHA + r")"),
    ("lote",              r"(?:lotes?|lt)\s*[:.]?\s*(.+?)(?=$|\s+[-–]\s*|\s*[-–]\s+)"),
    ("partida",           r"partida\s*[:.]?\s*(\S+)"),
    ("acta",              r"acta\s*[:.]?\s*(\S+)"),
    ("pallet",            r"pallets?\s*[:.]?\s*([\d\-\s]+)"),
    ("marca",             r"marca\s*[:.]?\s*(.+?)(?=$|\s+[-–]\s*|\s*[-–]\s+)"),
    ("orden_produccion",  r"\bop\s*[:.]?\s*([0-9][0-9/]*)"),
    ("articulo",          r"\bart\.?\s*[:.]?\s*([0-9]+)"),
    ("codigo",            r"c[oó]digo\s*[:.]?\s*(\S+)"),
    ("equipo",            r"equipo\s*[:.]?\s*(\S+(?:\s*F\s*\d+)?)"),
    ("silo",              r"silo\s*[:.]?\s*(\S+)"),
    ("big_bag",           r"\bbig(?:\s*bag)?\s*[:.]?\s*(\d+)"),
    ("bolsa",             r"bolsa\s*[:.]?\s*([\d\-\s]+)"),
    ("presentacion",      r"\bx\s*(\d+\s*(?:g|gr|kg|ml|l|cc)\b)"),
    ("hora",              r"(?:hora(?:rio)?|hs)\s*[:.]?\s*(" + HORA + r")"),
]
ETIQUETAS = [(n, re.compile(p, re.IGNORECASE)) for n, p in ETIQUETAS]

MUESTRA = re.compile(
    r"(?:^|[\s(\-–])(?:muestra\s*(?:n[°ºo]?)?\s*[:.]?\s*|M\s*)(\d{1,2})\b(?!\s*\d)",
    re.IGNORECASE,
)
COLA_MUESTRA = re.compile(r"\s+M\s*(\d{1,2})$", re.IGNORECASE)
HORA_SUELTA = re.compile(r"(?<![\d:/])(" + HORA + r")(?:\s*hs)?(?![\d:/])", re.IGNORECASE)

CAMPOS = [n for n, _ in ETIQUETAS] + ["muestra", "producto", "sin_identificar"]


def _limpiar(texto):
    """Normaliza espacios raros (incluido el no-separable \\xa0) y guiones largos."""
    texto = texto.replace("\xa0", " ").replace("\u2013", "-")
    return re.sub(r"\s+", " ", texto).strip()


def extraer_metadatos(descripcion):
    """Separa una Descripción en sus metadatos. Devuelve un dict con una clave
    por campo; lo que no se reconoce queda entero en 'sin_identificar'."""
    vacio = {c: None for c in CAMPOS}
    if not isinstance(descripcion, str) or not descripcion.strip():
        return vacio

    texto = _limpiar(descripcion)
    campos = dict(vacio)
    tramos = []  # (inicio, fin) de lo ya consumido, para no leerlo dos veces

    for nombre, patron in ETIQUETAS:
        m = patron.search(texto)
        if m and m.group(1):
            campos[nombre] = m.group(1).strip(" :.-")
            tramos.append(m.span())

    # un "M3" pegado al final del lote es el número de muestra, no parte del lote
    if campos["lote"]:
        m = COLA_MUESTRA.search(campos["lote"])
        if m:
            campos["lote"] = campos["lote"][: m.start()].strip()
            campos["muestra"] = "M" + m.group(1)

    def libre(pos):
        return not any(a <= pos < b for a, b in tramos)

    if campos["muestra"] is None:
        m = MUESTRA.search(texto)
        if m and libre(m.start()):
            campos["muestra"] = "M" + m.group(1)
            tramos.append(m.span())

    if campos["hora"] is None:
        for m in HORA_SUELTA.finditer(texto):
            if libre(m.start()):
                campos["hora"] = m.group(1)
                tramos.append(m.span())
                break

    # el producto es el primer tramo, siempre que no haya sido tomado ya por
    # una etiqueta (hay descripciones que arrancan directamente con "LOTE: ...")
    partes = SEPARADOR.split(texto)
    if partes:
        pos = texto.find(partes[0])
        if libre(pos):
            campos["producto"] = partes[0].strip(" :.-")
            tramos.append((pos, pos + len(partes[0])))

    # lo que quedó sin reconocer, en crudo
    resto = "".join(
        " " if any(a <= i < b for a, b in tramos) else ch for i, ch in enumerate(texto)
    )
    campos["sin_identificar"] = re.sub(r"[\s:\-–.]+", " ", resto).strip() or None
    return campos


def separar_descripcion(df, columna="Descripción"):
    """Devuelve un DataFrame con una columna por metadato, alineado con df."""
    meta = pd.DataFrame([extraer_metadatos(v) for v in df[columna]], index=df.index)
    return meta.reindex(columns=CAMPOS)

In [17]:
# Aplicación sobre el dataset consolidado y control de qué tan bien salió.

metadatos = separar_descripcion(SenasaFil)
SenasaFilMeta = pd.concat([SenasaFil, metadatos], axis=1)

print(f"Descripciones: {SenasaFil['Descripción'].notna().sum()} "
      f"({SenasaFil['Descripción'].nunique()} distintas)\n")

print(f"{'campo':20} {'con dato':>9} {'%':>7}")
for campo in CAMPOS:
    n = metadatos[campo].notna().sum()
    print(f"{campo:20} {n:9} {n / len(metadatos) * 100:6.1f}%")

# Lo que quedó sin reconocer: en su mayoría son marcas y nombres de cliente,
# que no se extraen a propósito. Lo que conviene mirar es el resto que TIENE
# números, porque ahí puede haber un metadato con un formato todavía no
# contemplado.
resto = metadatos["sin_identificar"].dropna()
con_numeros = resto[resto.str.contains(r"\d")]
print(f"\nSin identificar: {len(resto)} filas, de las cuales {len(con_numeros)} "
      f"tienen números (posibles metadatos no contemplados).")
print("\nLos 15 restos con números más frecuentes:")
print(con_numeros.value_counts().head(15).to_string())

Descripciones: 16347 (15453 distintas)

campo                 con dato       %
fecha_elaboracion          158    1.0%
fecha_vencimiento         1013    6.2%
fecha                      338    2.1%
lote                     12135   74.2%
partida                    592    3.6%
acta                       247    1.5%
pallet                     262    1.6%
marca                      106    0.6%
orden_produccion          1143    7.0%
articulo                   839    5.1%
codigo                     334    2.0%
equipo                     559    3.4%
silo                       112    0.7%
big_bag                    821    5.0%
bolsa                      276    1.7%
presentacion               601    3.7%
hora                      1643   10.1%
muestra                   3414   20.9%
producto                 16338   99.9%
sin_identificar           5842   35.7%

Sin identificar: 5842 filas, de las cuales 2902 tienen números (posibles metadatos no contemplados).

Los 15 restos con números más frecuent

## Etapa 6 — Conversión de los resultados a número

Hasta acá cada `Resultado` es **texto**, no número: las planillas traen la coma como separador
decimal (`46,35`), así que `read_excel` los lee como cadenas y ninguna de las 65 columnas de
resultado es numérica.

Eso rompe cualquier herramienta que espere números. dtale, por ejemplo, falla al pedirle un
histograma (`ufunc 'isfinite' not supported`), un promedio por categoría
(`agg function failed [how->mean, dtype->object]`) o una matriz de correlaciones, porque no
encuentra una sola columna numérica sobre la cual trabajar.

Hay tres formas que no convierten con sólo cambiar la coma por punto, y son el 10% de los valores:

| forma | qué significa | qué se hace |
|---|---|---|
| `&lt; 0,03`, `< 0,03` | por debajo del límite de detección del kit | **0** |
| `NO DETECTADO` | ídem | **0** |
| `6,90 +/- 0,04` | valor con incertidumbre de medición | se toma el valor |

Los `< 0,03` se computan como **0** y no como 0,03: la norma obliga a informarlos así, pero el
hecho medido es que no hay analito detectable. Son 3.780 valores, casi todos de lactosa y
galactosa, y tratarlos como 0,03 correría todas las medias hacia arriba.

El `&lt;` aparece escapado en HTML porque así viene del sistema de origen; hay que desescaparlo
antes de mirar el primer carácter.

In [18]:
# Conversión de las columnas 'Resultado' de texto a número.
#
# Devuelve además la condición de cada medición, para no perder la distinción
# entre "midió 0" y "está por debajo del límite de detección".

import html
import re

import numpy as np
import pandas as pd

INCERTIDUMBRE = re.compile(r"^(?P<valor>[\d.,]+)\s*(?:\+/-|±)\s*[\d.,]+$")


def a_numero(x):
    """Devuelve (valor, condición). '&lt; 0,03' -> (0.0, 'menor_que')."""
    if pd.isna(x):
        return (np.nan, "sin_dato")
    if isinstance(x, (int, float)):
        return (float(x), "medido")

    t = html.unescape(str(x)).strip()
    if not t:
        return (np.nan, "sin_dato")

    if t.upper().startswith(("NO DETECTADO", "NO CONTIENE", "AUSENCIA")) or t.upper() == "ND":
        return (0.0, "no_detectado")

    if t.startswith("<"):
        limite = pd.to_numeric(t.lstrip("<").strip().replace(",", "."), errors="coerce")
        # Por debajo del límite de detección: no hay analito, vale 0.
        return (0.0, "menor_que") if pd.notna(limite) else (np.nan, "texto")

    if t.startswith(">"):
        limite = pd.to_numeric(t.lstrip(">").strip().replace(",", "."), errors="coerce")
        return (float(limite), "mayor_que") if pd.notna(limite) else (np.nan, "texto")

    m = INCERTIDUMBRE.match(t)
    if m:
        v = pd.to_numeric(m.group("valor").replace(",", "."), errors="coerce")
        if pd.notna(v):
            return (float(v), "medido")

    v = pd.to_numeric(t.replace(",", "."), errors="coerce")
    return (float(v), "medido") if pd.notna(v) else (np.nan, "texto")


COLS_RESULTADO = [c for c in SenasaFilMeta.columns if c.startswith("Resultado")]

filas = []
for col in COLS_RESULTADO:
    parseado = SenasaFilMeta[col].map(a_numero)
    SenasaFilMeta[col] = [p[0] for p in parseado]
    condicion = pd.Series([p[1] for p in parseado])
    filas.append({
        "columna": col,
        "con_dato": int(SenasaFilMeta[col].notna().sum()),
        "censurados": int((condicion == "menor_que").sum()),
        "no_detectado": int((condicion == "no_detectado").sum()),
        "no_convertidos": int((condicion == "texto").sum()),
    })

resumen_conversion = pd.DataFrame(filas).sort_values("con_dato", ascending=False)

print(f"columnas convertidas: {len(COLS_RESULTADO)}")
print(f"columnas numéricas ahora: {len(SenasaFilMeta.select_dtypes('number').columns)}")
print(f"valores con dato: {resumen_conversion.con_dato.sum()}")
print(f"  censurados (< límite) computados como 0: {resumen_conversion.censurados.sum()}")
print(f"  no detectado computados como 0:          {resumen_conversion.no_detectado.sum()}")
print(f"  no convertibles (quedan vacíos):         {resumen_conversion.no_convertidos.sum()}")
print()
resumen_conversion.head(10)

columnas convertidas: 65
columnas numéricas ahora: 65
valores con dato: 40785
  censurados (< límite) computados como 0: 3780
  no detectado computados como 0:          46
  no convertibles (quedan vacíos):         33



,columna,con_dato,censurados,no_detectado,no_convertidos
41,Resultado Lactosa g /100g (*),9044,2591,4,1
3,Resultado Materia Grasa g/ 100g (*)_x,5838,15,0,0
0,Resultado Humedad g/ 100g (*),5659,0,0,0
6,Resultado Sólidos totales g/ 100g_x,5174,0,0,1
26,Resultado D - Galactosa g/ 100g (*),4583,804,0,0
2,Resultado Proteínas g/ 100g (*),3462,0,0,0
1,Resultado Cenizas g/ 100g,3061,0,0,0
4,Resultado Materia grasa en extracto seco g/ 10...,1634,0,0,0
10,Resultado Actividad de agua (% Aw) (*),611,0,0,0
17,Resultado Azucares totales g/100gr (*) (fructo...,204,35,0,0


### Qué mirar antes de analizar

`resumen_conversion` dice, por columna, cuántos de sus valores eran censurados. En lactosa son
2.591 sobre 9.044 y en galactosa 804 sobre 4.583: **más de un cuarto de la lactosa es 0 por estar
bajo el límite de detección**. Una media que los incluya no describe «la lactosa típica» sino
sobre todo cuántas muestras no tenían lactosa detectable, que es otra cosa.

Con esto el dataset ya se puede abrir en dtale:

```python
import dtale
dtale.show(SenasaFilMeta)
```